# 04 — External test on six new contracts

The GAT is trained on the six demo contracts and scored on six contracts in `test_contracts/`. Those contracts differ from the demo set in names, layout and compiler version. Only the arithmetic fix (compound assignments) was made after the first look at this test; see `FAILURE_LOG.md`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import platform, torch, networkx, sklearn, matplotlib
print("python", platform.python_version(), "| torch", torch.__version__,
      "| networkx", networkx.__version__, "| scikit-learn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)

python 3.13.3 | torch 2.13.0+cpu | networkx 3.6.1 | scikit-learn 1.9.0 | matplotlib 3.11.1


In [2]:
import evaluate_external
evaluate_external.main()

[*] Processing contracts to build GNN dataset...
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./demo_contracts\ReentrancyVulnerable.sol...
[+] Loaded ReentrancyVulnerable.sol (Nodes: 30, Edges: 54)
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./demo_contracts\ReentrancySafe.sol...


[+] Loaded ReentrancySafe.sol (Nodes: 32, Edges: 58)
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./demo_contracts\TimestampVulnerable.sol...
[+] Loaded TimestampVulnerable.sol (Nodes: 13, Edges: 22)
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./demo_contracts\TimestampSafe.sol...
[+] Loaded TimestampSafe.sol (Nodes: 19, Edges: 32)
[*] Detected pragma version requirement. Using solc 0.7.6
[*] Compiling ./demo_contracts\ArithmeticVulnerable.sol...


[+] Loaded ArithmeticVulnerable.sol (Nodes: 18, Edges: 18)
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./demo_contracts\ArithmeticSafe.sol...
[+] Loaded ArithmeticSafe.sol (Nodes: 43, Edges: 77)
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./test_contracts\VaultVulnerable.sol...


[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./test_contracts\VaultSafe.sol...
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./test_contracts\RaffleVulnerable.sol...
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./test_contracts\RaffleSafe.sol...


[*] Detected pragma version requirement. Using solc 0.7.6
[*] Compiling ./test_contracts\LedgerVulnerable.sol...
[*] Detected pragma version requirement. Using solc 0.8.20
[*] Compiling ./test_contracts\LedgerSafe.sol...
Test contracts: slicing and line check
  VaultVulnerable.sol      sliced nodes= 33 truth=[1.0, 0.0, 0.0] gold_lines=[14, 15]
  VaultSafe.sol            sliced nodes= 35 truth=[0.0, 0.0, 0.0] gold_lines=[]
  RaffleVulnerable.sol     sliced nodes= 22 truth=[0.0, 1.0, 0.0] gold_lines=[13]
  RaffleSafe.sol           sliced nodes= 18 truth=[0.0, 0.0, 0.0] gold_lines=[]
  LedgerVulnerable.sol     sliced nodes= 10 truth=[0.0, 0.0, 1.0] gold_lines=[12]
  LedgerSafe.sol           sliced nodes= 34 truth=[0.0, 0.0, 0.0] gold_lines=[]



External test: 18 decisions, 3 positive
  GAT (mean of 5 seeds)   P=1.000 R=1.000 F1=1.000 Acc=1.000
  Sink-presence rule      P=0.300 R=1.000 F1=0.462 Acc=0.611
  Logistic regression     P=0.000 R=0.000 F1=0.000 Acc=0.833
  GAT localization: mean precision=0.489, mean recall=0.667, n=15

GAT mean probability per decision:
  VaultVulnerable.sol      ['0.974', '0.000', '0.000']
  VaultSafe.sol            ['0.013', '0.000', '0.000']
  RaffleVulnerable.sol     ['0.000', '0.931', '0.000']
  RaffleSafe.sol           ['0.018', '0.004', '0.000']
  LedgerVulnerable.sol     ['0.000', '0.002', '0.999']
  LedgerSafe.sol           ['0.000', '0.000', '0.000']


## Results

The table below is read from `results/external_test/metrics.json`.

In [3]:
import json, pandas as pd
m = json.load(open("results/external_test/metrics.json"))
rows = {
    "GAT (mean of 5 seeds)": m["gat_mean"],
    "Sink-presence rule": {k: m["sink_rule"][k] for k in ["precision","recall","f1","accuracy"]},
    "Logistic regression": {k: m["logistic"][k] for k in ["precision","recall","f1","accuracy"]},
}
display(pd.DataFrame(rows).T.round(3))
print("GAT localization:", m["gat_localization"])

,precision,recall,f1,accuracy
GAT (mean of 5 seeds),1.0,1.0,1.000,1.000
Sink-presence rule,0.3,1.0,0.462,0.611
Logistic regression,0.0,0.0,0.000,0.833


GAT localization: {'mean_precision': 0.4888888888888889, 'mean_recall': 0.6666666666666666, 'rows': 15}


## Limits

Six contracts with three positives: one missed decision changes recall by 0.33. The contracts were written for this project, so this tests generalization across the authors' own styles, not real deployed code.